In [14]:
from pathlib import Path
import pandas as pd

metadata = pd.read_csv("data/metadata/tcga_brca_sample_level_dedup.csv")

count_files = list(Path("data/raw/star_counts").rglob("*.tsv"))
file_map = {p.name: p for p in count_files}

downloaded_filenames = set(file_map.keys())
metadata_filenames = set(metadata["file_name"])

len(downloaded_filenames), len(metadata_filenames)
missing_downloads = metadata_filenames - downloaded_filenames
extra_downloads = downloaded_filenames - metadata_filenames
len(downloaded_filenames), len(metadata_filenames), len(missing_downloads), len(extra_downloads)

(1231, 1219, 0, 12)

In [15]:
expected_gene_ids = None
count_series = []
gene_annotation = None

for i, row in metadata.iterrows():
    file_name = row["file_name"]
    sample_barcode = row["sample_barcode"]
    
    file_path = file_map[file_name]
    counts = pd.read_csv(file_path, sep="\t", comment="#")
    counts = counts.loc[~counts["gene_id"].str.startswith("N_")].copy()
    
    current_gene_ids = counts["gene_id"].tolist()
    
    if expected_gene_ids is None:
        expected_gene_ids = current_gene_ids
        gene_annotation = counts[["gene_id", "gene_name", "gene_type"]].copy()
    else:
        if current_gene_ids != expected_gene_ids:
            raise ValueError(f"Gene IDs/order do not match in file: {file_name}")
    
    sample_counts = counts.set_index("gene_id")["unstranded"]
    sample_counts.name = sample_barcode
    
    count_series.append(sample_counts)

count_matrix = pd.concat(count_series, axis=1)

In [16]:
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

count_matrix.to_csv(
    processed_dir / "tcga_brca_raw_unstranded_counts_sample_level.csv"
)

gene_annotation.to_csv(
    processed_dir / "tcga_brca_gene_annotation_from_star_counts.csv",
    index=False
)

In [17]:
count_matrix.shape
count_matrix.columns.duplicated().sum()
count_matrix.isna().sum().sum()
set(count_matrix.columns) == set(metadata["sample_barcode"])

True